
# Advanced LangGraph Parallel Workflow (Roman Urdu Edition)

This notebook is broken down into modular cells for easier understanding and reusability:
1. **Setup & Imports**
2. **State Definition**
3. **Node Functions** (Search, Drafting, Reviews, Editing)
4. **Graph Construction & Conditional Logic**
5. **Execution**
6. **Saving the Output**


In [ ]:

import os
import operator
from typing import TypedDict, Annotated, Sequence
from IPython.display import display, Image
import textwrap

from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage, BaseMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# --- 1. SETUP ---

# Initialize LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.5)

# Initialize research tools
web_tool = DuckDuckGoSearchRun()
wiki_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=1500)
wiki_tool = WikipediaQueryRun(api_wrapper=wiki_wrapper)

print("✅ Setup and tools initialized.")


In [ ]:

# --- 2. STATE DEFINITION ---

# Define the State schema that all nodes will pass around
class GraphState(TypedDict):
    topic: str
    web_research: str
    wiki_research: str
    draft: str
    style_review: str
    tech_review: str
    grammar_review: str
    editor_feedback: str
    final_article: str
    revision_count: int

print("✅ GraphState schema defined.")


In [ ]:

# --- 3. DEFINE NODES ---

def web_search_node(state: GraphState):
    print("-> \033[94mWeb Search Node\033[0m running...")
    topic = state['topic']
    try:
        results = web_tool.invoke(topic)
    except Exception as e:
        results = f"Web search failed: {e}"
    return {"web_research": results}

def wiki_search_node(state: GraphState):
    print("-> \033[94mWiki Search Node\033[0m running...")
    topic = state['topic']
    try:
        results = wiki_tool.invoke(topic)
    except Exception as e:
        results = f"Wiki search failed: {e}"
    return {"wiki_research": results}

def writer_node(state: GraphState):
    print("-> \033[92mWriter Node\033[0m running...")
    topic = state['topic']
    web_res = state.get('web_research', '')
    wiki_res = state.get('wiki_research', '')
    feedback = state.get('editor_feedback', '')
    rev_count = state.get('revision_count', 0)
    
    prompt = f"Write an informative article about: {topic}. The article MUST be written entirely in Roman Urdu.\n\n"
    prompt += f"Use this Web Research for context:\n{web_res}\n\n"
    prompt += f"Use this Wiki Research for context:\n{wiki_res}\n\n"
    
    if feedback:
        # If the editor sent it back, include the feedback!
        prompt += f"CRITICAL - Previous Draft Feedback to address:\n{feedback}\n\n"
        prompt += f"Current Draft to improve upon:\n{state.get('draft', '')}\n"
    
    response = llm.invoke([SystemMessage(content="You are an expert technical writer who writes fluently in Roman Urdu."), HumanMessage(content=prompt)])
    
    return {"draft": response.content, "revision_count": rev_count + 1}

def style_reviewer_node(state: GraphState):
    print("-> \033[93mStyle Reviewer\033[0m running...")
    prompt = f"Review the following Roman Urdu draft for tone, style, and flow. Is it engaging? Identify any awkward phrasing.\nDraft:\n{state['draft']}"
    response = llm.invoke([SystemMessage(content="You are a strict style and tone editor."), HumanMessage(content=prompt)])
    return {"style_review": response.content}

def tech_reviewer_node(state: GraphState):
    print("-> \033[93mTech Reviewer\033[0m running...")
    prompt = f"Review the following Roman Urdu draft for factual accuracy and technical depth based on the topic '{state['topic']}'. Point out any hallucinations or missing key facts.\nDraft:\n{state['draft']}"
    response = llm.invoke([SystemMessage(content="You are a ruthless technical fact-checker."), HumanMessage(content=prompt)])
    return {"tech_review": response.content}

def grammar_reviewer_node(state: GraphState):
    print("-> \033[93mGrammar Reviewer\033[0m running...")
    prompt = f"Review the following Roman Urdu draft for grammar, spelling, punctuation, and formatting. List any errors.\nDraft:\n{state['draft']}"
    response = llm.invoke([SystemMessage(content="You are a meticulous syntax editor."), HumanMessage(content=prompt)])
    return {"grammar_review": response.content}

def editor_node(state: GraphState):
    print("-> \033[95mEditor Synthesis Node\033[0m running...")
    draft = state.get('draft', '')
    style = state.get('style_review', '')
    tech = state.get('tech_review', '')
    grammar = state.get('grammar_review', '')
    
    prompt = f"You are the Editor-in-Chief. Evaluate this Roman Urdu draft based on the three reviews.\n\n"
    prompt += f"Draft:\n{draft}\n\n"
    prompt += f"Style Review:\n{style}\n\n"
    prompt += f"Tech Review:\n{tech}\n\n"
    prompt += f"Grammar Review:\n{grammar}\n\n"
    prompt += "If the draft has multiple critical flaws (especially technical errors or missing facts), return 'REVISE' followed by a detailed list of feedback the writer must fix.\n"
    prompt += "If the draft is generally good and only needs minor tweaks, rewrite it addressing the minor points maintaining the Roman Urdu language and return 'FINALIZE' followed by the rewritten Roman Urdu text.\n"
    prompt += "Your output MUST start with either 'REVISE:' or 'FINALIZE:'.\n"

    response = llm.invoke([SystemMessage(content="You are an Editor making a strict quality check."), HumanMessage(content=prompt)])
    content = response.content
    
    if content.startswith("REVISE"):
        feedback = content.replace("REVISE:", "").strip()
        print("   [EDITOR DECISION: \033[91mDraft Rejected - Needs Revision\033[0m]")
        # Clearing final article just in case
        return {"editor_feedback": feedback, "final_article": ""}
    else:
        final = content.replace("FINALIZE:", "").strip()
        print("   [EDITOR DECISION: \033[92mDraft Approved - Finalizing\033[0m]")
        return {"final_article": final, "editor_feedback": ""}

def finalize_node(state: GraphState):
    # Just a pass-through node to ensure the graph hits END cleanly
    return state

print("✅ All Nodes defined.")


In [ ]:

# --- 4. CONDITIONAL LOGIC AND GRAPH BUILDER ---

def check_quality(state: GraphState):
    # Route based on what the Editor produced in the final_article state
    if state.get("final_article"):
        return "finalize"
    elif state.get("revision_count", 0) >= 4:
        # Prevent infinite loops, force finalize after 4 revisions
        print("-> Max revisions (4) reached. Forcing completion.")
        return "finalize"
    else:
        return "revise"

# Build Graph Structure
builder = StateGraph(GraphState)

# Add Nodes
builder.add_node("web_search", web_search_node)
builder.add_node("wiki_search", wiki_search_node)
builder.add_node("writer", writer_node)
builder.add_node("style_reviewer", style_reviewer_node)
builder.add_node("tech_reviewer", tech_reviewer_node)
builder.add_node("grammar_reviewer", grammar_reviewer_node)
builder.add_node("editor", editor_node)
builder.add_node("finalize_node", finalize_node)

# START -> RESEARCH (Parallel)
builder.add_edge(START, "web_search")
builder.add_edge(START, "wiki_search")

# RESEARCH -> WRITER (Fan-in wait: writer runs when both are done)
builder.add_edge("web_search", "writer")
builder.add_edge("wiki_search", "writer")

# WRITER -> REVIEWS (Fan-Out 3)
builder.add_edge("writer", "style_reviewer")
builder.add_edge("writer", "tech_reviewer")
builder.add_edge("writer", "grammar_reviewer")

# REVIEWS -> EDITOR (Fan-in wait: editor runs when all three are done)
builder.add_edge("style_reviewer", "editor")
builder.add_edge("tech_reviewer", "editor")
builder.add_edge("grammar_reviewer", "editor")

# Conditional Loop
builder.add_conditional_edges(
    "editor",
    check_quality,
    {
        "revise": "writer",
        "finalize": "finalize_node"
    }
)

# finalize to END
builder.add_edge("finalize_node", END)

# Compile
graph = builder.compile()

print("✅ Graph built and compiled.")

# (Optional) Display Graph Image if running in a real notebook environment
try:
    png_bytes = graph.get_graph().draw_mermaid_png()
    display(Image(png_bytes))
except Exception as e:
    print("Could not display graph map:", e)


In [ ]:

# --- 5. RUN THE WORKFLOW ---

initial_state = {
    "topic": "The exact impact of AI agents on future programming jobs",
    "revision_count": 0
}

print(f"Starting Workflow for topic: '{initial_state['topic']}'\n")

# Run the graph
final_state = graph.invoke(initial_state)

print("\n" + "="*50)
print("\033[1mFINAL ARTICLE PUBLISHED:\033[0m")
print("="*50 + "\n")

# Determine what content to save based on if it passed editing
content_to_save = ""
if final_state.get('final_article'):
    content_to_save = final_state['final_article']
    print(content_to_save)
else:
    # Fallback if it maxed out revisions but Editor didn't finalize properly
    content_to_save = final_state.get('draft', 'No draft generated.')
    print("Failed to pass editorial review in given attempts. Printing last draft:\n")
    print(content_to_save)


In [ ]:

# --- 6. SAVE TO FILE ---
# You can customize the filename here
output_filename = "ai_agents_impact.txt" 

try:
    with open(output_filename, "w", encoding="utf-8") as f:
        # We can add a nice header to the text file too
        f.write(f"TOPIC: {initial_state['topic']}\n")
        f.write(f"REVISIONS: {final_state.get('revision_count', 0)} out of max 4\n")
        f.write("LANGUAGE: Roman Urdu\n")
        f.write("="*50 + "\n\n")
        
        # Word wrap the text file output for easier reading
        wrapped_content = textwrap.fill(content_to_save, width=80)
        f.write(wrapped_content)
        
    print(f"\n✅ Successfully saved the result to '{output_filename}' in the current folder.")
except Exception as e:
    print(f"\n❌ Failed to save the file: {e}")
